# Foraging Cache — DuckDB Query Examples

The production parquet database lives on S3:
- `s3://aind-scratch-data/aind-dynamic-foraging-cache/session_table.parquet` — flat session table (one row per session)
- `.../trial_table/` — Hive-partitioned by `subject_id` (one row per trial)
- `.../event_table/` — Hive-partitioned by `subject_id` (one row per behavioral event)

DuckDB reads `s3://` natively and the cache bucket is **public** — no AWS credentials or setup are required. The paths (`SESSION_DB`, `TRIAL_DB`, `EVENT_DB`) are importable from `aind_dynamic_foraging_data_utils.foraging_cache`; reassign them to a local directory to query a local build instead.

**Primary pattern**: filter the session table, then JOIN to the trial/event tables so every row carries the session-level metadata (subject, date, task, `foraging_eff`, …) alongside the trial data.

Three options when reading the partitioned tables:
- `hive_partitioning=true` — partition-level pruning on `subject_id`
- `union_by_name=true` — merges column schemas across files (different NWB readers produce different column sets; missing columns fill with NULL)
- `CAST(subject_id AS VARCHAR)` — DuckDB infers the `subject_id=<n>` directory name as BIGINT; cast to VARCHAR to compare against the session table

In [ ]:
import time

import duckdb

from aind_dynamic_foraging_data_utils.foraging_cache import SESSION_DB, TRIAL_DB, EVENT_DB

# SESSION_DB / TRIAL_DB / EVENT_DB point at the public production cache on S3.
# Reassign them to a local directory to query a local build instead.

# Reusable snippets - union_by_name handles schema differences across NWB reader paths.
READ_TRIALS = f"read_parquet('{TRIAL_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

## 1. Database at a glance

A quick orientation to the whole cache. Session/mice counts come from one cheap scan of the flat session table; total trials and the timed load scan the partitioned trial table.

In [ ]:
# Sessions, mice, and date span — one scan of the flat session table
duckdb.sql(f"""
    SELECT
        COUNT(*)                   AS total_sessions,
        COUNT(DISTINCT subject_id) AS total_mice,
        MIN(session_date)          AS first_session,
        MAX(session_date)          AS last_session
    FROM read_parquet('{SESSION_DB}')
""").df()

In [ ]:
# Distribution of data sources (which NWB route produced each session)
duckdb.sql(f"""
    SELECT nwb_data_source,
           COUNT(*)                   AS n_sessions,
           COUNT(DISTINCT subject_id) AS n_mice
    FROM read_parquet('{SESSION_DB}')
    GROUP BY nwb_data_source
    ORDER BY n_sessions DESC
""").df()

In [ ]:
# Total trials across the entire database (scans the partitioned trial table)
duckdb.sql(f"SELECT COUNT(*) AS total_trials FROM {READ_TRIALS}").df()

### Load the 5-column behavioral projection for **every** trial, and time it

Column projection (choice / reward / reward-probabilities) keeps this fast and light even over the full ~12.5M-trial database on S3 — the normal starting point for a population analysis.

In [ ]:
t0 = time.time()
df_5col = duckdb.sql(f"""
    SELECT session_id, animal_response, earned_reward,
           reward_probabilityL, reward_probabilityR
    FROM {READ_TRIALS}
""").df()
elapsed = time.time() - t0

print(f"Loaded {len(df_5col):,} trials x {df_5col.shape[1]} columns "
      f"from {df_5col['session_id'].nunique():,} sessions "
      f"in {elapsed:.1f}s ({df_5col.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory)")
df_5col.head()

## 2. Browse the session table

In [ ]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_DB}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

## 3. Primary use case — filter sessions -> trials with session keys merged

Filter the session table however you like, then JOIN to the trial table so every trial row already carries the session-level metadata.

In [ ]:
df_trials = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        t.session_id,
        t.trial,
        t.animal_response,
        t.earned_reward,
        t.reward_probabilityL,
        t.reward_probabilityR,
        t.rewarded_historyL,
        t.rewarded_historyR
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date
""").df()

print(f"{len(df_trials):,} trials from {df_trials['session_id'].nunique():,} sessions")
df_trials.head(10)

## 4. Same pattern for events

In [ ]:
df_events = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        e.session_id,
        e.timestamps,
        e.event,
        e.data
    FROM {READ_EVENTS} e
    JOIN sel s ON e.session_id = s._session_id
    WHERE CAST(e.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date, e.timestamps
""").df()

print(f"{len(df_events):,} events from {df_events['session_id'].nunique():,} sessions")
print(f"Event types: {sorted(df_events['event'].unique())}")
df_events.head(10)

## 5. Aggregate across sessions — all in SQL

In [ ]:
duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        COUNT(DISTINCT s._session_id)  AS n_sessions,
        COUNT(*)                       AS n_trials,
        AVG(t.earned_reward::DOUBLE)   AS mean_reward_rate,
        AVG(s.foraging_eff)            AS mean_foraging_eff
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    GROUP BY s.subject_id
    ORDER BY mean_foraging_eff DESC
""").df()